In [1]:
import editor

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
# settings
array_size = 256
num_channels = 3

nn_intake_fov = 9
nn_hidden_dim = 128

run_device = torch.device("cuda")

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # self.conv1 = nn.Conv2d(num_channels, 6, 15)
        # self.conv2 = nn.Conv2d(6, 6, 9)
        # self.conv3 = nn.Conv2d(6, 9, 6)
        # self.conv4 = nn.Conv2d(9, 9, 3)
        # self.conv5 = nn.Conv2d(9, 9, 2)

        self.fc1 = nn.Linear(nn_intake_fov * nn_intake_fov * num_channels, nn_hidden_dim)
        self.fc2 = nn.Linear(nn_hidden_dim, nn_hidden_dim)
        self.fc3 = nn.Linear(nn_hidden_dim, nn_hidden_dim)
        self.fc4 = nn.Linear(nn_hidden_dim, num_channels)
        
        self.bn1 = nn.BatchNorm1d(num_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x = F.relu(self.conv1(x))
        # x = F.relu(self.conv2(x))
        # x = F.relu(self.conv3(x))
        # x = F.relu(self.conv4(x))
        # x = F.relu(self.conv5(x))

        x = x.flatten(start_dim=1)

        # print(x.size())

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.bn1(F.sigmoid(self.fc4(x)))

        return x

In [4]:
class Trainer:
    def __init__(self, model):
        self.model = model

    def train(self, world: torch.Tensor, region: tuple[int, int, int, int],
              lr: float, epochs: int) -> float:
        """Train the model on one region of the world.

        world:  (C, H, W) full world, region: (y0, x0, y1, x1), end-exclusive.
        Every cell in the region should map to its current value (the region stays as it is).
        Returns the last loss.
        """
        y0, x0, y1, x1 = region
        C, H, W = world.shape
        k = nn_intake_fov
        lo, hi = (k - 1) // 2, k // 2

        world = world.detach().clone().to(run_device)  # plain tensor, safe for autograd

        # the region plus its real surroundings (wrapping at the world edges)
        ys = torch.arange(y0 - lo, y1 + hi, device=run_device) % H
        xs = torch.arange(x0 - lo, x1 + hi, device=run_device) % W
        context = world[:, ys][:, :, xs]

        h, w = y1 - y0, x1 - x0
        batch = F.unfold(context[None], k).transpose(1, 2).reshape(h * w, C, k, k)  # one slice per cell
        target = world[:, y0:y1, x0:x1].reshape(C, -1).T                           # (h*w, C)

        optimizer = optim.AdamW(self.model.parameters(), lr=lr)
        was_training = self.model.training
        self.model.train()

        for _ in range(epochs):
            optimizer.zero_grad()
            loss = ((self.model(batch) - target) ** 2).mean()
            loss.backward()
            optimizer.step()

        self.model.train(was_training)
        return loss.item()


In [5]:
class Simulator:
    def __init__(self, model):
        self.model = model
        self.trainer = Trainer(self.model)

    def step(self, state: torch.Tensor):
        state = state.to(run_device)

        C, H, W = state.shape
        k = nn_intake_fov
        lo, hi = (k - 1) // 2, k // 2
        x = F.pad(state[None], (lo, hi, lo, hi), mode="circular")
        slices = F.unfold(x, k).transpose(1, 2).reshape(H * W, C, k, k)  # one slice per cell
        out = self.model(slices)                                         # (H*W, outputs per slice)

        out = out.cpu()

        return out.reshape(H, W, -1).permute(2, 0, 1)                    # (outputs per slice, H, W)

    def train(self, world, region, lr, epochs):
        return self.trainer.train(world, region, lr, epochs)

In [6]:
model = Net(nn_intake_fov).to(run_device)

In [7]:
sim = Simulator(model)

In [8]:
editor = editor.NCAEditor(sim, torch.rand((num_channels, array_size, array_size)))

In [9]:
editor.run()